In [4]:
#!/usr/bin/env python3
"""
Take the first and last posterior samples from a run_emcee_walker.py
results folder's posteriors.csv, build a simulated Population from each,
and plot them side by side against the Pluto reference population for an
easy before/after comparison of the chain.

Usage:
    python plot_posterior_result.py results/Pluto_test/Pluto_test_.../posteriors.csv
"""
import argparse
import csv
import os
import pandas as pd

import commentjson

import param_versions
import population_class
from params import det_prob
from population_class import Population, Pluto
from population_plotter import plot_sep_vs_dm_comparison, plot_pa_vs_sep_comparison


def load_first_and_last_sample(posteriors_path):
    """
    Read a posteriors.csv in a single pass and return (first_sample,
    last_sample), each a dict mapping parameter name to value.
    """
    with open(posteriors_path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        first_row = next(reader)
        reader = reversed(list(csv.reader(f)))
        last_row = next(reader)
    if first_row is None:
        raise ValueError(f"No posterior samples found in {posteriors_path}")

    def to_dict(row):
        return dict(zip(header, (float(v) for v in row)))

    return to_dict(first_row), to_dict(last_row)


def load_params_version(posteriors_path):
    """
    If a runprops.txt sits alongside posteriors.csv (as run_emcee_walker.py
    leaves in a run's results folder) and names a params_version, swap in
    that Parameters/<version>.py snapshot so the Population we build here
    matches what the run actually used.
    """
    runprops_path = os.path.join(os.path.dirname(posteriors_path), "runprops.txt")
    if not os.path.exists(runprops_path):
        return
    with open(runprops_path) as f:
        runprops = commentjson.load(f)
    params_version = runprops.get("params_version")
    if params_version:
        param_versions.apply_params_version(params_version, [globals(), vars(population_class)])
        print(f"Using parameter model '{params_version}' from Parameters/")


def plot_first_last_comparison(posteriors_path, reference_pop, det_prob_fn=det_prob):
    """
    Build simulated Populations from the first and last rows of
    `posteriors_path` and plot them next to each other (and against
    `reference_pop`) so the start and end of the chain are easy to compare.

    Returns (first_pop, last_pop).
    """
    first_params, last_params = load_first_and_last_sample(posteriors_path)
    first_pop = Population(reference_pop.popu, first_params, det_prob_fn)
    last_pop = Population(reference_pop.popu, last_params, det_prob_fn)

    named_pops = [("First posterior", first_pop), ("Last posterior", last_pop)]
    plot_sep_vs_dm_comparison(named_pops, reference_pop.popu)
    plot_pa_vs_sep_comparison(named_pops, reference_pop.popu)

    return first_pop, last_pop


def main(posteriors_path):
    load_params_version(posteriors_path)
    reference_pop = Pluto()
    print(reference_pop)

    first_pop, last_pop = plot_first_last_comparison(posteriors_path, reference_pop)

    print("First posterior sample:")
    print(first_pop)
    print("Last posterior sample:")
    print(last_pop)


if __name__ == "__main__":
    main('results/Pluto_test/Pluto_test_2026-08-06_15.44.21_000/params.py')

Using parameter model 'params_1_0' from Parameters/
Reference population
 Singles: 44
 Moonlike: 456
 Wide: 0
 Total: 500
 Binary fraction: 0.912
 Log likelihood: N/A



ValueError: could not convert string to float: '    def mu(self): return self.mu_fn()'